In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from datetime import date

In [3]:
from prophet import Prophet
from prophet.diagnostics import performance_metrics
from prophet.plot import plot_cross_validation_metric
from prophet.diagnostics import cross_validation

c:\Users\Windows 10\Desktop\rosman-store-project\forecast-sales\.salesvenv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Importing plotly failed. Interactive plots will not work.


In [27]:
df = pd.read_csv('./data/train.csv')
df['Date'] = pd.to_datetime(df['Date'])

C:\Users\Windows 10\AppData\Local\Temp\ipykernel_9608\2732287264.py:1: DtypeWarning: Columns (0: StateHoliday) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('./data/train.csv')


In [29]:
agg_df.head()

,Date,Sales
0,2013-01-01,97235
1,2013-01-02,6949829
2,2013-01-03,6347820
3,2013-01-04,6638954
4,2013-01-05,5951593


In [ ]:
# 2013-01-01 00:00:00 inicio
# 2015-07-31 00:00:00 fim

In [21]:
forecast_start_date = (date(2015,7,1)).strftime("%Y-%m-%d") 
print(forecast_start_date)

prediction_days = 30

2015-07-01


In [30]:
df_copy = agg_df.copy()
df_copy = df_copy.rename(columns={'Date': 'ds', 'Sales': 'y'})
df_copy[['y']] = df_copy[['y']].apply(pd.to_numeric)

train_set = df_copy[(df_copy['ds'] < forecast_start_date) ]

In [38]:
m = Prophet()
m.fit(train_set)
df_cv = cross_validation(m, initial='365 days', period='30 days', horizon = '30 days')

18:27:24 - cmdstanpy - INFO - Chain [1] start processing
18:27:24 - cmdstanpy - INFO - Chain [1] done processing
Seasonality has period of 365.25 days which is larger than initial window. Consider increasing initial.
  0%|          | 0/18 [00:00<?, ?it/s]18:27:26 - cmdstanpy - INFO - Chain [1] start processing
18:27:26 - cmdstanpy - INFO - Chain [1] done processing
  6%|▌         | 1/18 [00:00<00:08,  1.89it/s]18:27:26 - cmdstanpy - INFO - Chain [1] start processing
18:27:26 - cmdstanpy - INFO - Chain [1] done processing
 11%|█         | 2/18 [00:00<00:07,  2.06it/s]18:27:27 - cmdstanpy - INFO - Chain [1] start processing
18:27:27 - cmdstanpy - INFO - Chain [1] done processing
 17%|█▋        | 3/18 [00:01<00:07,  2.13it/s]18:27:27 - cmdstanpy - INFO - Chain [1] start processing
18:27:27 - cmdstanpy - INFO - Chain [1] done processing
 22%|██▏       | 4/18 [00:01<00:06,  2.15it/s]18:27:28 - cmdstanpy - INFO - Chain [1] start processing
18:27:28 - cmdstanpy - INFO - Chain [1] done process

In [39]:
df_p = performance_metrics(df_cv)

In [46]:
df_p.head(30)

,horizon,mse,rmse,mae,mape,mdape,smape,coverage
0,3 days,4.351563e+12,2.086040e+06,1.599667e+06,1.062247,0.204595,0.360113,0.740741
1,4 days,5.402902e+12,2.324414e+06,1.681373e+06,1.692092,0.200105,0.425149,0.703704
2,5 days,5.667292e+12,2.380608e+06,1.734462e+06,1.733268,0.208132,0.406924,0.722222
3,6 days,5.453316e+12,2.335234e+06,1.638391e+06,1.699417,0.246857,0.412130,0.796296
4,7 days,3.679384e+12,1.918172e+06,1.436377e+06,1.130214,0.230143,0.350475,0.851852
5,8 days,2.382424e+12,1.543510e+06,1.244594e+06,0.469148,0.208081,0.328227,0.851852
6,9 days,2.011840e+12,1.418393e+06,1.140838e+06,0.429958,0.186297,0.311346,0.870370
7,10 days,1.828681e+12,1.352287e+06,1.068341e+06,0.517870,0.153540,0.312311,0.888889
8,11 days,1.745719e+12,1.321257e+06,9.960596e+05,0.513859,0.132554,0.323126,0.907407
9,12 days,2.696238e+12,1.642023e+06,1.117999e+06,1.123471,0.139563,0.322459,0.870370


In [ ]:
# print(df_cv['ds'].min())
# print(df_cv['ds'].max())

# 2014-01-07 00:00:00
# 2015-06-30 00:00:00

2014-01-07 00:00:00
2015-06-30 00:00:00


In [43]:
df_cv['mape'] = (df_cv['y']-df_cv['yhat'])/(df_cv['y'])*100
df_cv['overestimate'] = df_cv['yhat'] > df_cv['y'] 

df_cv.sort_values('mape',ascending=False).head(10)

,ds,yhat,yhat_lower,yhat_upper,y,cutoff,mape,overestimate
362,2015-01-04,-1.387213e+06,-3.668544e+06,7.979432e+05,197193,2015-01-01,803.479999,False
369,2015-01-11,-1.251774e+06,-3.672602e+06,9.205480e+05,183367,2015-01-01,782.660389,False
5,2014-01-12,-1.026388e+06,-3.312730e+06,1.285275e+06,175744,2014-01-06,684.024252,False
376,2015-01-18,-7.229591e+05,-2.948391e+06,1.505511e+06,195778,2015-01-01,469.274925,False
12,2014-01-19,-6.109279e+05,-2.940540e+06,1.822644e+06,175778,2014-01-06,447.556538,False
306,2014-11-09,-5.713088e+05,-2.895870e+06,1.569935e+06,177774,2014-11-02,421.368008,False
271,2014-10-05,-6.319584e+05,-2.960303e+06,1.725619e+06,225796,2014-10-03,379.880234,False
278,2014-10-12,-5.646801e+05,-3.129680e+06,1.551162e+06,240423,2014-10-03,334.869430,False
383,2015-01-25,-3.777970e+05,-2.757002e+06,1.963323e+06,179028,2015-01-01,311.026763,False
285,2014-10-19,-3.927855e+05,-2.762459e+06,1.793634e+06,204991,2014-10-03,291.611094,False
